
![DBAcademy](https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/icons/databricks_academy.png)

# Lecture - Data Ingestion from Cloud Storage

## Overview

In this lecture, you will learn how raw files from cloud storage can be efficiently converted into Delta tables using Databricks tools, unlocking advanced management and analytics capabilities within the Lakehouse.


## Learning Objectives

By the end of this lecture, you will be able to:

1. **Demonstrate how to ingest data from cloud object storage into Delta tables** using CREATE TABLE AS, COPY INTO, and Auto Loader, including **capturing input file metadata in Bronze layer tables**
2. **Explain how rescued columns are used during ingestion** to manage malformed records

## A. Data Ingestion Patterns From Cloud Object Storage

Data ingestion is a critical component of modern Lakehouse architecture, enabling organizations to take advantage of large volumes of data stored in cloud object storage systems.


<div style="max-width: 800px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<div style="display: flex; gap: 16px; align-items: center;">

  <!-- Cloud Storage -->
  <div style="flex: 0 0 170px; background: #F9F7F4; border-radius: 10px; padding: 18px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center;">
    <div style="font-size: 15pt; font-weight: 700; margin-top: 10px;">Cloud Storage</div>
    <ul style="font-size: 14pt; padding-left: 18px; margin: 10px 0 0; text-align: left;">
      <li>CSV</li>
      <li>JSON</li>
      <li>Parquet</li>
      <li>etc.</li>
    </ul>
  </div>

  <!-- Arrow -->
  <div style="font-size: 28pt; color: #618794;"> > </div>

  <!-- Data Ingestion Methods -->
  <div style="flex: 0 0 240px; background: #4299E0; border-radius: 10px; padding: 18px; text-align: center; color: white;">
    <div style="font-size: 15pt; font-weight: 700; margin-bottom: 14px;">Data Ingestion</div>
    <div style="display: flex; flex-direction: column; gap: 8px;">
      <div style="background: white; color: #1B5162; border-radius: 6px; padding: 8px; font-size: 14pt; font-weight: 500;">CREATE TABLE AS</div>
      <div style="background: white; color: #1B5162; border-radius: 6px; padding: 8px; font-size: 14pt; font-weight: 500;">COPY INTO</div>
      <div style="background: white; color: #1B5162; border-radius: 6px; padding: 8px; font-size: 14pt; font-weight: 500;">AUTO LOADER</div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="font-size: 28pt; color: #618794;"> > </div>

  <!-- Delta Table -->
  <div style="flex: 0 0 170px; background: #F9F7F4; border-radius: 10px; padding: 18px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center;">
    <div style="font-size: 15pt; font-weight: 700; margin-top: 10px;">Delta Table</div>
    <img src="https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/icons/table.png" style="height: 150px;">
  </div>

</div>

<!-- Bottom callout -->
<div style="
  margin: 18px auto 0;
  padding: 12px 20px;
  background: #F9F7F4;
  border: 2px solid #4299E0;
  border-radius: 8px;
  text-align: center;
  font-size: 14pt;
  max-width: 900px;
">
  Convert <strong>raw file formats</strong> to <strong>Delta tables</strong>
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES
<details>
<ul>
<li>Common file formats like <strong>CSV, JSON, and Parquet</strong> are frequently used due to their flexibility and ease of use.</li>
<li>Our goal is to convert these <strong>raw files into Delta tables</strong>, unlocking advanced functionality such as ACID transactions, time travel, and schema enforcement.</li>
<li>We'll explore three <strong>primary methods for ingesting files</strong> from cloud object storage into Delta tables:
<ul>
<li>CREATE TABLE AS (CTAS)</li>
<li>COPY INTO</li>
<li>Auto Loader</li>
</ul>
</li>
<li>Ingestion from cloud object storage is performed using <strong>Lakeflow Connect Standard Connectors</strong>.</li>
</ul>
</details>

## B. Data Ingestion Methods

When ingesting data into Databricks using Lakeflow Connect Standard Connectors, you can choose from several ingestion methods.

##### Click on the tabs to switch between the ingestion methods.


<div style="width: 100%; margin: auto; font-family: sans-serif;">

<style>
.four-grid {
    display: flex;
    flex-direction: column;
    gap: 60px;
    justify-content: center;
    align-items: center;
}
.ing-box {
    width: 80%;
    min-height: 400px;
    background: #F9F7F4;
    border: none;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    overflow: hidden;
    display: flex;
    flex-direction: column;
    gap: 12px;
    padding: 20px;
    text-align: center;
    position: relative;
    box-sizing: border-box;
}
.ing-box::before {
    content: "";
    position: absolute;
    top: 0; left: 0;
    width: 100%; height: 8px;
}
.ing-box.batch::before       { background: #2574B5; }
.ing-box.incremental::before { background: #02A36F; }
.ing-box.streaming::before   { background: #FE3722; }

.ing-box-title { font-size: 16pt; font-weight: bold; text-align: center; }
.ing-box-icon  { display: inline-flex; align-items: center; justify-content: center; gap: 8px; }
.ing-box-icon img {
    width: 50px; height: auto;
    background: transparent;
    mix-blend-mode: multiply;
    filter: contrast(1.15) brightness(1);
    border-radius: 4px;
}

.ing-box-content {
    display: flex;
    align-items: flex-start;
    justify-content: center;
    gap: 24px;
    width: 100%;
}
.ing-box-text {
    font-size: 14pt;
    max-width: 500px;
    text-align: left;
    line-height: 1.6;
}
.ing-box-text ul { text-align: left; padding-left: 18px; margin: 0 0 14px 0; }
.ing-box-text li { margin-bottom: 10px; }
.ing-box-text li:last-child { margin-bottom: 0; }
.ing-example {
    padding: 12px 14px;
    border-radius: 8px;
    font-size: 14pt;
    line-height: 1.6;
    font-weight: 400;
}
.batch .ing-example       { background: rgba(37,116,181,0.10); border-left: 4px solid #2574B5; }
.incremental .ing-example { background: rgba(2,163,111,0.10);  border-left: 4px solid #02A36F; }
.streaming .ing-example   { background: rgba(254,55,34,0.10);  border-left: 4px solid #FE3722; }

.code-block-wrapper {
    flex: 1;
    min-width: 100;
    min-height: 200px;
}

.code-block-container {
    background: #f8f8f8;
    border-radius: 8px;
    padding: 16px;
    overflow-x: auto;
    border: 1px solid #e0e0e0;
    font-family: Consolas, Monaco, monospace;
    font-size: 10pt;
    line-height: 1.6;
    text-align: left;
}
</style>

<!-- Tabs -->
<div style="display: flex; border-bottom: 2px solid #EEEDE9; margin-bottom: 0;">
  <button class="dbtab" onclick="showTab(1)" style="padding: 10px 18px; border: none; border-bottom: 3px solid #2574B5; background: none; font-size: 14pt; font-weight: bold; color: #2574B5; cursor: pointer; margin-bottom: -2px;">CREATE TABLE AS</button>
  <button class="dbtab" onclick="showTab(2)" style="padding: 10px 18px; border: none; border-bottom: 3px solid transparent; background: none; font-size: 14pt; font-weight: bold; color: #888; cursor: pointer; margin-bottom: -2px;">COPY INTO</button>
  <button class="dbtab" onclick="showTab(3)" style="padding: 10px 18px; border: none; border-bottom: 3px solid transparent; background: none; font-size: 14pt; font-weight: bold; color: #888; cursor: pointer; margin-bottom: -2px;">AUTO LOADER</button>
</div>

<br>

<!-- TAB 1 -->
<div class="dbpanel" style="display: block;">
  <div class="four-grid">
    <div class="ing-box batch">
      <div class="ing-box-title">
        <div class="ing-box-icon">
          <img src="https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/icons/batch.png" alt="Create Table As">
          <span>Method 1 - Batch - <br><code>CREATE TABLE AS (CTAS)</code></span>
        </div>
      </div>
      <div class="ing-box-content">
        <div class="code-block-wrapper">
        <div class="code-block" data-language="sql">
          CREATE TABLE new_table AS
          SELECT *
          FROM read_files(
            &lt;<i>path_to_file(s)</i>&gt;,
            format => '&lt;<i>file_type</i>&gt;',
            &lt;<i>other_format_specific_options</i>&gt;
          );
        </div>
        </div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>
<br>
<div class="ing-box-text">
  <div class="ing-example">
<code><strong>CREATE TABLE AS (CTAS)</strong></code> creates a Delta table <strong>by default</strong> from files in cloud object storage.
</div>
<br>
  <div class="ing-example">
The <code><strong>read_files()</strong></code> function reads files under a provided location and returns the data in <strong>tabular form.</strong>
  </div>
        </div>
      </div>
    </div>
  </div>
</div>


<!-- TAB 2 -->
<div class="dbpanel" style="display: none;">
  <div class="four-grid">
    <div class="ing-box incremental">
      <div class="ing-box-title">
        <div class="ing-box-icon">
          <img src="https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/icons/incremental.png" alt="COPY INTO">
          <span>Method 2 - Incremental Batch - <br><code>COPY INTO</code></span>
        </div>
      </div>
      <div class="ing-box-content">
        <div class="code-block-wrapper">
        <div class="code-block" data-language="sql">
        CREATE TABLE new_table;
        <br>
        COPY INTO new_table
        FROM '&lt;<i>dir_path</i>&gt;'
        FILEFORMAT = &lt;<i>file_type</i>&gt;
        FORMAT_OPTIONS (&lt;<i>options</i>&gt;)
        COPY_OPTIONS (&lt;<i>options</i>&gt;)
        </div>
        </div>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>
<br>
<div class="ing-box-text">
<div class="ing-example">
Use the <code><strong>COPY INTO</strong></code> statement to copy files from cloud storage into the Delta table; this performs a bulk load from files in cloud object storage into the table.
</div>
<br>
<div class="ing-example">
The <code><strong>COPY INTO</strong></code> will skip any files that have already been loaded into the table, and only new files will be ingested.
</div>
        </div>
      </div>
    </div>
  </div>
</div>

<!-- TAB 3 -->
<div class="dbpanel" style="display: none;">
  <div class="four-grid">
    <div class="ing-box streaming">
      <div class="ing-box-title">
        <div class="ing-box-icon">
          <img src="https://files.training.databricks.com/binder/prod_main/data-ingestion-with-lakeflow-connect-en_us-3.1.1/images/20260731T171356Z/Data Ingestion with LakeFlow Connect/Includes/images/icons/streaming.png" alt="AUTO LOADER">
          <span>Method 3 - Incremental Batch or Streaming - <br><code>AUTO LOADER</code></span>
        </div>
      </div>
      <div class="ing-box-content">
        <div style="max-width: 700px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
          <div style="display: flex; gap: 20px; align-items: flex-start;">
            <!-- Python -->
            <div style="flex: 1;">
              <div style="font-size: 12pt; font-weight: 400; margin-bottom: 10px;"><strong>Python Auto Loader</strong></div>
              <div class="code-block-container">
                <div class="code-block-wrapper">
                  (spark<br>
                  &nbsp;&nbsp;.readStream<br>
                  &nbsp;&nbsp;&nbsp;&nbsp;.<span style="color: #4299E0;">format</span>(<span style="color: #00A972;">"cloudFiles"</span>)<br>
                  &nbsp;&nbsp;&nbsp;&nbsp;.<span style="color: #4299E0;">option</span>(<span style="color: #00A972;">"cloudFiles.format"</span>, <span style="color: #00A972;">"json"</span>)<br>
                  &nbsp;&nbsp;&nbsp;&nbsp;.<span style="color: #4299E0;">option</span>(<span style="color: #00A972;">"cloudFiles.schemaLocation"</span>, <span style="color: #00A972;">"&lt;<code>checkpoint_path</code>&gt;"</span>)<br>
                  &nbsp;&nbsp;&nbsp;&nbsp;.<span style="color: #4299E0;">load</span>(<span style="color: #00A972;">"/Volumes/catalog/schema/files"</span>)<br>
                  &nbsp;&nbsp;.writeStream<br>
                  &nbsp;&nbsp;&nbsp;&nbsp;.<span style="color: #4299E0;">option</span>(<span style="color: #00A972;">"checkpointLocation"</span>, <span style="color: #00A972;">"&lt;<code>checkpoint_path</code>&gt;"</span>)<br>
                  &nbsp;&nbsp;&nbsp;&nbsp;.<span style="color: #4299E0;">trigger</span>(processingTime=<span style="color: #00A972;">"5 seconds"</span>)<br>
                  &nbsp;&nbsp;&nbsp;&nbsp;.<span style="color: #4299E0;">toTable</span>(<span style="color: #00A972;">"catalog.database.table"</span>)<br>
                  )
                </div>
              </div>
            </div>
            <!-- SQL -->
            <div style="flex: 1;">
              <div style="font-size: 12pt; font-weight: 400; margin-bottom: 10px;"><strong>Auto Loader with SQL (Declarative Pipelines)</strong></div>
              <div class="code-block-container">
                <span style="color: #4299E0; font-weight: 400;">CREATE OR REFRESH STREAMING TABLE</span><br>
                &nbsp;&nbsp;catalog.schema.table<br>
                <span style="color: #4299E0; font-weight: 400;">SCHEDULE EVERY</span> 1 HOUR<br>
                <span style="color: #4299E0; font-weight: 400;">AS</span><br>
                <span style="color: #4299E0; font-weight: 400;">SELECT</span> *<br>
                <span style="color: #4299E0; font-weight: 400;">FROM STREAM</span> read_files(<br>
                &nbsp;&nbsp;'&lt;<code>dir_path</code>&gt;',<br>
                &nbsp;&nbsp;format => '&lt;<code>file_type</code>&gt;'<br>
                )
              </div>
            </div>
          </div>
        </div>
      </div>
    </div>
  </div>
</div>

<script>
function showTab(n) {
  var tabs   = document.getElementsByClassName("dbtab");
  var panels = document.getElementsByClassName("dbpanel");
  var colors = ["#2574B5", "#02A36F", "#FE3722"];
  for (var i = 0; i < tabs.length; i++) {
    tabs[i].style.color        = "#888";
    tabs[i].style.borderBottom = "3px solid transparent";
    panels[i].style.display    = "none";
  }
  tabs[n-1].style.color        = colors[n-1];
  tabs[n-1].style.borderBottom = "3px solid " + colors[n-1];
  panels[n-1].style.display    = "block";
}
window.onload = function() { showTab(1); };
</script>

##### Documentation

For more information on Auto Loader, see the official [Databricks documentation](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader) and [tutorials](https://www.databricks.com/resources/demos/tutorials?itm_data=demo_center).

##### EXPAND FOR ADDITIONAL NOTES
<details>
<ul>
<li><strong>CREATE TABLE AS (CTAS)</strong>
  <ol>
    <li>Supports reading <strong>file formats</strong> like:<br>
        | JSON | CSV | XML | TEXT | BINARYFILE | PARQUET | AVRO | ORC</li>
    <li>Can detect the file format automatically and <strong>infer a unified schema</strong> across all files.</li>
    <li>Specify <strong>specific file format options</strong> to read in the data based on the source file format.</li>
    <li>Can be used in <strong>streaming tables</strong> to <strong>incrementally</strong> ingest files into Delta Lake using Auto Loader.</li>
  </ol>
</li><br>
<li><strong>COPY INTO</strong><br>
Use the <code><strong>COPY INTO</strong></code> statement to copy files from cloud storage into the Delta table. This command performs a bulk load from files in cloud object storage into the table, and in this example, it will load files into the empty table new_table. The <code><strong>FROM</strong></code> clause specifies the location of the CSV files.</li><br>
<code><strong>COPY INTO</strong></code> is ideal for situations where the cloud storage location is continuously adding files, since it is a retriable and idempotent operation designed for incremental batch ingestion.

  <strong>Key aspects of <code>COPY INTO</code></strong>:
  <ul>
    <li><strong>Idempotent</strong>: Will skip any files that have already been loaded into the table; only new files will be ingested</li>
    <li><strong>File format support</strong>: Parquet, JSON, XML, and others</li>
    <li><strong><code>FROM clause</code></strong>: Specifies the path of the cloud storage location where new files are being continuously added</li>
    <li><strong><code>FORMAT_OPTIONS()</code></strong>: Controls how the source files are parsed and interpreted (options depend on file format)</li>
    <li><strong><code>COPY_OPTIONS()</code></strong>: Controls the behavior of the <code>COPY INTO</code> operation itself, such as:
      <ul>
        <li>Schema evolution using (<strong>mergeSchema</strong>)</li>
        <li>Idempotency using (<strong>force</strong>)</li>
      </ul>
    </li>
  </ul>
</li><br>
<li><strong>AUTO LOADER</strong>
  <ol>
    <li>Incremental batch or streaming ingestion using Auto Loader.
      <ul>
        <li>Process new data files <strong>incrementally</strong> as they arrive in cloud storage (batch or streaming)</li>
        <li>Ingest data <strong>without extra setup</strong> or complex configuration</li>
        <li><strong>Automatically</strong> detect and load new files into Delta tables</li>
        <li>Simplify handling of incremental and streaming data</li>
        <li>Use with both <strong>Python</strong> and <strong>SQL</strong> (via Declarative Pipelines)</li>
        <li>Scale to <strong>process billions of files</strong></li>
        <li>Rely on <strong>Spark Structured Streaming</strong> for efficient and reliable ingestion</li>
      </ul>
    </li><br>
    <li>Auto Loader in Python to read streaming data from cloud storage:
      <ul>
        <li>We start with <strong><code>.readStream</code></strong> and set the <strong>format</strong> to "<code><strong>cloudFiles</strong></code>", which enables Auto Loader.</li>
        <li>Then, we specify the <strong>file format</strong> as <strong>JSON</strong>, and define the <strong>schema location</strong> using <code><strong>cloudFiles.schemaLocation</strong></code>, which is used to track schema inference and evolution.</li>
        <li>Next, we use <strong><code>.load()</code></strong> to point to the location of the files, in this case a path under /Volumes referencing Unity Catalog.</li>
        <li>On the write side, we configure <strong><code>.writeStream</code></strong> with a <strong>checkpoint location</strong> to maintain state and progress, and set a <strong>trigger</strong> interval of <strong>every 5 seconds</strong>.</li>
        <li>Finally, we use <strong><code>.toTable()</code></strong> to write the data into a Delta table specified by catalog, database, and table name.</li>
      </ul>
    </li><br>
    <li>Auto Loader with Databricks SQL
      <ul>
        <li>Databricks recommends using streaming tables to ingest data with Databricks SQL (instead of COPY INTO). A streaming table is a table registered to <strong>Unity Catalog</strong> that includes additional support for streaming or incremental data processing.
          <ul>
            <li>When you create a streaming table, a <strong>pipeline</strong> is automatically generated for it.</li>
            <li>Streaming tables can be used for incremental data loading from both <strong>Kafka</strong> and <strong>cloud object storage</strong>.</li>
          </ul>
        </li>
        <li>To create a streaming table from files in a volume, you use Auto Loader. Databricks recommends using <strong>Auto Loader with Apache Spark™ Declarative Pipelines</strong> for most data ingestion tasks from cloud object storage. Together, Auto Loader and Declarative Pipelines are designed to incrementally and idempotently load continuously growing datasets as they arrive.</li>
        <li>Streaming tables in Databricks SQL are backed by <strong>serverless</strong> Spark Declarative Pipelines. Your workspace must support serverless pipelines to use this functionality. Alternatively, you can <strong>build your own</strong> Spark Declarative Pipelines for incremental processing, optimization, and monitoring. Declarative Pipelines offer a range of additional features, which you can learn more about <a href="https://docs.databricks.com/aws/en/dlt/" style="color:#1976D2;">here</a>.</li>
        <li>To use Auto Loader in Databricks SQL, use the <strong><code>read_files</code></strong> function with the <strong><code>STREAM</code></strong> keyword in the <strong><code>FROM</code></strong> clause.</li>
      </ul>
    </li>
  </ol>
</li>
</ul>
</details>

## C. Ingestion Methods at a Glance

Here is a quick summary of all three data ingestion methods.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style>
table td, table th {
  font-size: 12pt !important;
}
table ul {
  font-size: 12pt !important;
}
</style>

<table style="width: 100%; border-collapse: collapse; line-height: 1.5;">
  <thead>
    <tr style="background: #1B5162; color: white;">
      <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9; width: 140px;">FEATURE</th>
      <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">CREATE TABLE AS (CTAS) + spark.read</th>
      <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">COPY INTO</th>
      <th style="padding: 10px 14px; text-align: left; border: 1px solid #EEEDE9;">Auto Loader</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background: #F9F7F4;">
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700;">Ingestion Type</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Batch</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Incremental Batch</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Incremental (Batch or Streaming)</td>
    </tr>
    <tr>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700;">Use Cases</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Best for smaller datasets</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Ideal for thousands of files</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Scale to millions+ of files per hour, backfills with billions of files</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700;">Syntax/Interface</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">
        <ul style="margin: 0; padding-left: 16px;">
          <li>Python (spark.read)</li>
          <li>SQL (CTAS)</li>
        </ul>
      </td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">SQL</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">
        <ul style="margin: 0; padding-left: 16px;">
          <li>Python (spark.readStream)</li>
          <li>SQL with Declarative Pipelines (CREATE OR REFRESH STREAMING TABLES)</li>
          <li>Streaming tables in Databricks SQL</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700;">Idempotency</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">No</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Yes</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Yes</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700;">Schema Evolution</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Manual or inferred during read</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Supported with options</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Auto Loader automatically detects and evolves schemas. Handles new columns as they appear.</td>
    </tr>
    <tr>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700;">Latency</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">High</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Moderate (scheduled)</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Low or high depending on configuration</td>
    </tr>
    <tr style="background: #F9F7F4;">
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700;">Ease of Use</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Simple</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Simple and SQL-based</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Intermediate to advanced depending on the implementation</td>
    </tr>
    <tr>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9; font-weight: 700;">Summary</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Best for one-time, ad hoc ingestion. Can be scheduled to always read and process all data.</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Simple and repeatable for incremental file ingestion. Great for scheduled jobs or pipelines.</td>
      <td style="padding: 10px 14px; border: 1px solid #EEEDE9;">Best for near real-time streaming or incremental ingestion, with high automation and scalability.</td>
    </tr>
  </tbody>
</table>

</div>

## D. Conclusion

In this lecture, you learned the three primary methods for ingesting data from cloud object storage into Delta tables:

- **`CREATE TABLE AS (CTAS)`**: Batch ingestion using `read_files()` that creates Delta tables from raw files. Best for smaller, ad hoc datasets.
- **`COPY INTO`**: Incremental batch ingestion that is idempotent and retriable. Skips already-loaded files and supports format and copy options for fine-grained control.
- **`AUTO LOADER`**: The most scalable method, built on Spark Structured Streaming. Supports both Python and SQL (via Declarative Pipelines), processes billions of files, and automatically handles schema evolution.

### Next Steps

In the next section, you will work hands-on with these ingestion methods to load data from cloud storage into Delta tables.

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/>
<a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> |
<a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> |
<a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>